# Stage 3 - Clean / prepare data

Order: **redact** (own logs only) → **normalise** → **filters** (language, toxicity, ≥2 exchanges and relevance for WildChat) → **near-duplicate removal** → **segments**.

Reads `raw/*.jsonl` from Stage 2. Writes `filtered/<source>.jsonl`, `segments/<source>.jsonl` and `filtered/cleaning_report.json`. Settings: `configs/clean.yaml`.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[clean]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.clean.pipeline import run_cleaning

report = run_cleaning("clean")

In [ ]:
import pandas as pd

# Counts after every step, per source - this table goes in the report
pd.DataFrame(report).T.fillna(0).astype(int)

## Hand check of the relevance filter

1. Run the next cell: it writes 200 random kept WildChat conversations to a CSV on Drive.
2. Open it, and fill the `is_relevant` column with `y` or `n` (is this really about programming or requirements?).
3. Run the cell after it to get the precision with a 95% interval. If it is low, a zero-shot classifier gets added to the filter.

In [ ]:
from krs.clean.precision_check import export_sample
from krs.config import get_paths

check_csv = get_paths().filtered / "precision_check_wildchat.csv"
export_sample(get_paths().filtered / "wildchat.jsonl", check_csv, n=200)
print(check_csv)

In [ ]:
from krs.clean.precision_check import precision

precision(check_csv)   # run after filling is_relevant

In [ ]:
# Look at a few segments
from krs.io import read_jsonl
from krs.schema import Segment

for s, _ in zip(read_jsonl(get_paths().segments / "swe_agent.jsonl", Segment), range(2)):
    print(s.segment_id, '| project:', s.project_id)
    print('CONTEXT:', s.context_text[:300], '\nTEXT:', s.text[:500], '\n')

## Own logs - local only

If `raw/own_logs.jsonl` exists (collected locally with `run_own_logs`), it is redacted first. Add names of people, clients and internal systems to `own_logs/redact_terms.txt` (one per line, gitignored), then skim the redacted segments before labelling: rules cannot catch every name.